# 05 - Model Development
===

Anomaly detection and conflict prediction models.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({'figure.dpi': 150, 'figure.figsize': (10, 6)})
sns.set_style('whitegrid')

DATA_DIR = Path('./data/processed')
OUTPUT_DIR = Path('./outputs/models')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
df = pd.read_parquet(DATA_DIR / 'ais_features.parquet')
df_sample = df.sample(n=min(50000, len(df)), random_state=42
print(f"Training records: {len(df_sample):,}")

## Anomaly Detection

In [ ]:
from src.models.anomaly_model import AnomalyDetector

anomaly_detector = AnomalyDetector(str(OUTPUT_DIR / 'anomaly'))
anomaly_results = anomaly_detector.run(df_sample, contamination=0.05)

print(f"Anomalies detected: {anomaly_results['anomaly_label'].sum():,}")
print(f"Anomaly types: {anomaly_results['anomaly_type'].value_counts().head()}")

## Conflict Prediction

In [ ]:
from src.models.conflict_predictor import ConflictPredictor

predictor = ConflictPredictor(str(OUTPUT_DIR / 'predictor'))
pred_results = predictor.run(df_sample, test_size=0.2)

print("\n=== Model Evaluation ===")
for model_name, metrics in pred_results.items():
    if isinstance(metrics, dict):
        print(f"{model_name}: AUROC={metrics.get('auroc', 0):.3f}, F1={metrics.get('f1', 0):.3f}")

## Model Evaluation

In [ ]:
from src.models.evaluator import ModelEvaluator

# Evaluate best model
eval = ModelEvaluator(str(OUTPUT_DIR / 'evaluator'))

# Generate comparison
fig, ax = plt.subplots(figsize=(10, 5))
model_names = list(pred_results.keys())
auroc_scores = [pred_results[m].get('auroc', 0) for m in model_names if isinstance(pred_results[m], dict)]
f1_scores = [pred_results[m].get('f1', 0) for m in model_names if isinstance(pred_results[m], dict)]

x = np.arange(len(auroc_scores))
width = 0.35
ax.bar(x - width/2, auroc_scores, width, label='AUROC', color='steelblue')
ax.bar(x + width/2, f1_scores, width, label='F1', color='coral')
ax.set_xticks(x)
ax.set_xticklabels([m.replace('_', '\n') for m in model_names if isinstance(pred_results[m], dict)], fontsize=8)
ax.set_ylabel('Score')
ax.set_title('Model Comparison')
ax.legend()
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'model_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

## Summary

In [ ]:
print("="*50)
print("MODEL DEVELOPMENT COMPLETE")
print("="*50)
print(f"Anomaly model: {OUTPUT_DIR / 'anomaly'}")
print(f"Prediction model: {OUTPUT_DIR / 'predictor'}")
print(f"Evaluations: {OUTPUT_DIR / 'evaluator'}")